In [15]:
import osmnx as ox
import pandas as pd
import networkx as nx
import geopandas as gpd

## Import noise street segments

In [16]:
noise_streets = gpd.read_file("../../layers/BCN_noise_streets.gpkg")
print(noise_streets.crs)  #CRS = Coordinate Reference System
print(noise_streets.shape)
print(noise_streets.columns.tolist())
print("Number of street segments:", len(noise_streets))

EPSG:25831
(15115, 30)
['TRAM', 'TOTAL_D', 'TOTAL_E', 'TOTAL_N', 'TOTAL_DEN', 'TRANSIT_D', 'TRANSIT_E', 'TRANSIT_N', 'TRANSIT_DEN', 'GI_TR_D', 'GI_TR_E', 'GI_TR_N', 'GI_TR_DEN', 'FFCC_D', 'FFCC_E', 'FFCC_N', 'FFCC_DEN', 'INDUST_D', 'INDUST_E', 'INDUST_N', 'INDUST_DEN', 'VIANANTS_D', 'VIANANTS_E', 'OCI_N', 'PATIS_D', 'PATIS_E', 'geometry_type', 'start', 'end', 'geometry']
Number of street segments: 15115


## Compute centrality

IMPORTANT! It takes long to run

Betwenness centrality based on edges is the strongest predictor of noise, as it is related to traffic flow.

In [17]:
G = ox.graph_from_place("Barcelona, Spain", network_type="drive")
G = ox.distance.add_edge_lengths(G)

In [18]:
ebc = nx.edge_betweenness_centrality(G, weight='length', normalized=True)
nx.set_edge_attributes(G, ebc, 'edge_betweenness')

nodes, edges = ox.graph_to_gdfs(G)

## Spatial join with noise streets

In [19]:
edges_copy = edges.to_crs(noise_streets.crs)

noise_streets = noise_streets.sjoin(
    edges_copy[['edge_betweenness', 'geometry']],
    how='left',
    predicate='intersects'
)

noise_streets = noise_streets.groupby(noise_streets.index).agg({
    'edge_betweenness': 'max',
    'geometry': 'first',
    'TRAM': 'first',
})

noise_streets.head(10)

,edge_betweenness,geometry,TRAM
0,0.000501,"MULTILINESTRING ((430229.789 4586585.199, 4301...",T04719W
1,NaN,"MULTILINESTRING ((432928.898 4584019.988, 4329...",T19941Z
2,0.000110,"MULTILINESTRING ((429953.263 4588161.441, 4299...",T18111R
3,0.000457,"MULTILINESTRING ((427949.038 4580946.043, 4279...",T03222Y
4,0.000238,"MULTILINESTRING ((433950.189 4585741.214, 4338...",T17625I
5,0.000943,"MULTILINESTRING ((429464.339 4587309.389, 4295...",T05360P
6,0.001085,"MULTILINESTRING ((431912.281 4585068.507, 4318...",T08863T
7,0.000315,"MULTILINESTRING ((432088.266 4581318.697, 4321...",T00236S
8,0.003190,"MULTILINESTRING ((431196.393 4590356.601, 4312...",T13009A
9,0.000349,"MULTILINESTRING ((431707.97 4588314.797, 43165...",T11921P


## Create dataset

In [20]:
dataset = pd.DataFrame({
    "street_id": noise_streets['TRAM'],
})
dataset['edge_betweenness'] = noise_streets['edge_betweenness']

## Export as CSV

In [22]:
import os
output_dir = "../../data/processed"
os.makedirs(output_dir, exist_ok=True)
dataset.to_csv(os.path.join(output_dir, "networkx_streets.csv"), index=False)
print("Exported networkx_streets.csv")

Exported networkx_streets.csv


In [ ]:
"""
noise_streets['centroid'] = noise_streets.geometry.centroid

cc = nx.closeness_centrality(G, distance='length')
nx.set_node_attributes(G, cc, 'closeness')

bc = nx.betweenness_centrality(G, weight='length', normalized=True)
nx.set_node_attributes(G, bc, 'betweenness')

noise_streets = noise_streets.sjoin_nearest(
    nodes[['closeness', 'geometry']],
    how='left',
    distance_col='dist_to_node'
)

noise_streets = noise_streets.sjoin_nearest(
    nodes[['betweenness', 'geometry']],
    how='left'
)


TODO: add straightness
"""